# Track Deleted Locations (oxjob #850)

Maintains the durable ledger of ES location docs whose id no longer exists in
`openalex.works.locations_mapped`. Same design as TrackDeletedWorks (#784): runs
right after Guardrails, so a run that loses locations pathologically is stopped
before it can ledger the loss.

The unit here is the **ES doc id** `native_id_namespace:native_id`, not the
table's 3-part primary key (provenance, namespace, native_id): the sync collapses
provenance into one doc per id, so a doc may be deleted only when NO eligible row
still produces that id. "Eligible" = the exact filter `sync_locations` applies
(native_id + namespace present, work_id present and > 0) — a row that becomes
ineligible (e.g. loses its work id) drops out of the set and its doc is ledgered
for deletion. Keep the two filters byte-identical.

Two tables:

- **`openalex.works.deleted_locations`** — the ledger. One row per (doc id,
  deletion): `id` (STRING, the ES `_id`), `deleted_date` (first date the id was
  detected gone), `deleted_at` (timestamp), `es_deleted_at` (stamped by
  `notebooks/elastic/delete_locations` once the ES doc is removed; NULL = ES
  delete still pending).
- **`openalex.works.locations_es_id_snapshot`** — the eligible-id set as of the
  last successful run of this notebook. Tonight's deletions = snapshot ∖ current.
  Rewritten at the end of every run, only after the ledger insert succeeded.
  First run seeds it and ledgers nothing.

**Resurrections**: any id alive in the current eligible set is dropped from the
ledger — a live doc must never be listed as deleted. The revived doc restores
itself in ES without help: a returning row has no `locations_mapped_hash` entry
(that table is rebuilt from `locations_mapped` nightly, so the vanish removed
its row), so the rebuild stamps a fresh `openalex_updated_dt` and the watermark
sync re-indexes the doc.

**Guard**: aborts (no ledger insert, snapshot untouched) if the eligible set is
empty or tonight's deletions exceed `guard_fraction` (default 0.5%, ~3.3M docs)
of the live count. Sanctioned mass deletions (#765/#880-class waves) run with
the `deleted_locations_guard_override` job parameter set to `true`.

No registry/tombstone purge here — location identity policy lives in
TrackDeletedWorks (work_id_map / location_work_ids purge).


In [0]:
dbutils.widgets.text("guard_override", "false")
dbutils.widgets.text("guard_fraction", "0.005")
dbutils.widgets.text("env_suffix", "")

GUARD_OVERRIDE = dbutils.widgets.get("guard_override").lower() == "true"
GUARD_FRACTION = float(dbutils.widgets.get("guard_fraction"))
ENV_SUFFIX = dbutils.widgets.get("env_suffix")

CATALOG = f"openalex{ENV_SUFFIX}"
MAPPED = f"{CATALOG}.works.locations_mapped"
LEDGER = f"{CATALOG}.works.deleted_locations"
ID_SNAPSHOT = f"{CATALOG}.works.locations_es_id_snapshot"
ELIGIBLE = f"{CATALOG}.works._tmp_eligible_location_ids_850"

print(f"guard_override: {GUARD_OVERRIDE}")
print(f"guard_fraction: {GUARD_FRACTION}")
print(f"mapped: {MAPPED}")


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LEDGER} (
    id STRING NOT NULL,
    deleted_date DATE NOT NULL,
    deleted_at TIMESTAMP NOT NULL,
    es_deleted_at TIMESTAMP
)
""")


In [0]:
# Materialize the eligible-id set ONCE (the DISTINCT over ~740M rows is the
# expensive part; everything below reuses it). Filter must stay byte-identical
# to notebooks/elastic/sync_locations.
spark.sql(f"""
CREATE OR REPLACE TABLE {ELIGIBLE} AS
SELECT DISTINCT CONCAT(native_id_namespace, ':', native_id) AS id
FROM {MAPPED}
WHERE native_id IS NOT NULL AND native_id_namespace IS NOT NULL
  AND work_id IS NOT NULL AND work_id > 0
""")

resurrected = spark.sql(
    f"DELETE FROM {LEDGER} WHERE id IN (SELECT id FROM {ELIGIBLE})"
).collect()[0].num_affected_rows
print(f"Resurrected location ids removed from ledger: {resurrected:,}")


In [0]:
current_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {ELIGIBLE}").collect()[0].cnt
print(f"Current eligible location ids: {current_count:,}")

if not spark.catalog.tableExists(ID_SNAPSHOT):
    if current_count == 0:
        raise Exception(f"ABORT: eligible set from {MAPPED} is empty; refusing to seed {ID_SNAPSHOT} from it.")
    spark.sql(f"CREATE TABLE {ID_SNAPSHOT} AS SELECT id FROM {ELIGIBLE}")
    print(f"First run: seeded {ID_SNAPSHOT} with {current_count:,} ids; no deletions ledgered.")
else:
    gone_df = spark.sql(f"""
        SELECT s.id
        FROM {ID_SNAPSHOT} s
        LEFT ANTI JOIN {ELIGIBLE} e ON s.id = e.id
    """)
    gone_count = gone_df.count()
    print(f"Location ids gone since last run: {gone_count:,}")

    if current_count == 0:
        raise Exception(
            f"ABORT: eligible set from {MAPPED} is empty; every id would ledger as deleted. "
            "Upstream build failed in a way Guardrails missed. Ledger and snapshot unchanged."
        )
    if gone_count > GUARD_FRACTION * current_count and not GUARD_OVERRIDE:
        raise Exception(
            f"ABORT: {gone_count:,} location ids gone (> {GUARD_FRACTION:.2%} of {current_count:,} live). "
            "Ledger and snapshot unchanged, so tonight's diff is preserved for inspection. "
            "If this is a sanctioned mass deletion, re-run with deleted_locations_guard_override=true."
        )

    gone_df.createOrReplaceTempView("gone_location_ids")
    inserted = spark.sql(f"""
        INSERT INTO {LEDGER}
        SELECT g.id, current_date(), current_timestamp(), NULL
        FROM gone_location_ids g
        LEFT ANTI JOIN {LEDGER} l ON g.id = l.id
    """).collect()[0].num_inserted_rows
    print(f"Ledgered {inserted:,} deletions (dated {spark.sql('SELECT current_date() AS d').collect()[0].d}).")

    spark.sql(f"CREATE OR REPLACE TABLE {ID_SNAPSHOT} AS SELECT id FROM {ELIGIBLE}")
    print(f"Snapshot rewritten with {current_count:,} ids.")

spark.sql(f"DROP TABLE IF EXISTS {ELIGIBLE}")

pending = spark.sql(
    f"SELECT COUNT(*) AS cnt FROM {LEDGER} WHERE es_deleted_at IS NULL"
).collect()[0].cnt
total = spark.sql(f"SELECT COUNT(*) AS cnt FROM {LEDGER}").collect()[0].cnt
print(f"Ledger: {total:,} total deletions, {pending:,} pending ES delete.")
